# Spatial Spiking Information (SSI) Walkthrough

This notebook is meant to be stepped through slowly. It starts with a single-frame activation map so the SSI calculation is visible, then connects the same quantities to the cached BackImage digital-twin summaries.

The key idea: even when we plot one current frame, the model activations can carry history. A recurrent/history-dependent twin has seen a retinal movie leading up to this frame, so two identical current images can still yield different activation maps under static, empirical, scaled empirical, Brownian, or rotated gaze histories.

## 1. Setup

Run this cell first after opening the notebook from the repository root. The real-twin demo section uses a small self-cache: the first run loads the checkpoint and writes activation maps, and later runs reuse that cache.

In [ ]:
from __future__ import annotations

from pathlib import Path
import os
import sys
import math
import re

os.environ.setdefault("MPLCONFIGDIR", "/tmp/matplotlib-cache")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import Markdown, display

plt.rcParams.update({
    "figure.dpi": 130,
    "savefig.dpi": 200,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "font.size": 9,
})

def find_repo_root(start: Path | None = None) -> Path:
    start = Path.cwd() if start is None else Path(start)
    for candidate in [start, *start.parents]:
        if (candidate / "pyproject.toml").exists() and (candidate / "VisionCore").exists():
            return candidate
    return start

REPO_ROOT = find_repo_root()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

print(REPO_ROOT)

## 2. Analysis Knobs

The requested explanatory comparison is: static/no motion, empirical 0.5x, empirical 1x, empirical 2x, Brownian, and rotated. The cache section below uses the same labels.

In [ ]:
N_UNITS_TO_SHOW = 12
DT = 1.0 / 120.0
RNG_SEED = 7

REQUESTED_CONDITIONS = [
    ("static", "static", "static"),
    ("empirical 0.5x", "empirical", "rel_0p5x"),
    ("empirical 1x", "empirical", "rel_1x"),
    ("empirical 2x", "empirical", "rel_2x"),
    ("brownian 1x", "brownian", "rel_1x"),
    ("rotated 1x", "rotated", "rel_1x"),
]

CACHE_CANDIDATES = [
    REPO_ROOT / "outputs/fixation_statistics_by_stimulus_all_sessions_after_review/backimage_aggregate_fem_information_ssi_combined_n128_K2_rel025-2_k16_seed0/response_summary_arrays.npz",
    REPO_ROOT / "outputs/fixation_statistics_by_stimulus_all_sessions_after_review/backimage_aggregate_fem_information_ssi_combined_n64_rel025-2_k16_seed0/response_summary_arrays.npz",
    REPO_ROOT / "outputs/fixation_statistics_by_stimulus_all_sessions_after_review/backimage_aggregate_fem_information_ssi_combined_pilot_n16_rel05_1_k4_seed0/response_summary_arrays.npz",
]

for path in CACHE_CANDIDATES:
    print(("found" if path.exists() else "missing"), path.relative_to(REPO_ROOT))

## 3. SSI Formula

For a unit with spatial rate map `r(x)` over positions `x`, define the spatial mean rate `rbar = E_x[r(x)]` and gain `g(x) = r(x) / rbar`.

`SSI_unit = E_x[ g(x) log2(g(x)) ]` in bits/spike.

For a population, we usually spike-weight units: units with larger expected spike count contribute more to the population bits/spike summary.

In [ ]:
def spatial_ssi_single_frame(rate_maps: np.ndarray, eps: float = 1e-8) -> dict[str, np.ndarray | float]:
    """SSI for one frame of spatial maps with shape (unit, height, width)."""
    y = np.asarray(rate_maps, dtype=np.float64)
    if y.ndim != 3:
        raise ValueError(f"Expected (unit, height, width), got {y.shape}")
    if np.any(y < 0):
        raise ValueError("Rate maps must be non-negative.")

    n_units, height, width = y.shape
    flat = y.reshape(n_units, height * width)
    rbar = flat.mean(axis=1)
    gain = flat / (rbar[:, None] + eps)
    local_bits = gain * np.log2(gain + eps)
    unit_bits_per_spike = local_bits.mean(axis=1)

    weights = rbar / np.maximum(rbar.sum(), eps)
    pop_bits_per_spike = float(np.sum(weights * unit_bits_per_spike))
    return {
        "unit_bits_per_spike": unit_bits_per_spike,
        "unit_mean_rate": rbar,
        "gain": gain.reshape(n_units, height, width),
        "local_bits": local_bits.reshape(n_units, height, width),
        "population_bits_per_spike": pop_bits_per_spike,
    }

def spatial_ssi_movie(rate_map: np.ndarray, dt: float = DT, eps: float = 1e-8) -> dict[str, np.ndarray]:
    """Project-compatible SSI for shape (time, unit, height, width)."""
    y = np.asarray(rate_map, dtype=np.float64)
    if y.ndim != 4:
        raise ValueError(f"Expected (time, unit, height, width), got {y.shape}")
    t_max, n_units, height, width = y.shape
    flat = y.reshape(t_max, n_units, height * width)
    rbar = flat.mean(axis=2)
    gain = flat / (rbar[..., None] + eps)
    unit_bits = np.mean(gain * np.log2(gain + eps), axis=2)
    expected_spikes = rbar * float(dt)
    bits = expected_spikes * unit_bits
    bits_per_spike = bits.sum(axis=1) / np.maximum(expected_spikes.sum(axis=1), eps)
    return {
        "bits_per_spike": bits_per_spike.astype(np.float32),
        "unit_bits_per_spike": unit_bits.astype(np.float32),
        "unit_mean_rate": rbar.astype(np.float32),
        "bits_per_bin": bits.sum(axis=1).astype(np.float32),
        "expected_spikes": expected_spikes.sum(axis=1).astype(np.float32),
    }

## 4. Real Twin Activation Maps

This section computes the actual digital-twin activation maps for one natural image frame under the requested history/control trajectories. The first run loads the twin checkpoint and writes a small cache under `notebooks/cache/`; later runs read the cache and skip the model call.

In [ ]:
RUN_REAL_TWIN_DEMO = os.environ.get("SSI_WALKTHROUGH_SKIP_REAL_TWIN", "0") != "1"
FORCE_REAL_TWIN_RECOMPUTE = False
REAL_TWIN_CACHE_PATH = REPO_ROOT / "notebooks/cache/spatial_ssi_real_twin_demo_rate_maps.npz"
REAL_TWIN_IMAGE_INDEX = 24
REAL_TWIN_T_MAX = 40
REAL_TWIN_FRAME_INDEX = REAL_TWIN_T_MAX - 1
REAL_TWIN_BATCH_SIZE = 8
REAL_TWIN_DEVICE = None  # set to "cpu", "cuda:0", etc. to override get_free_device()

print("real twin demo:", "enabled" if RUN_REAL_TWIN_DEMO else "skipped")
print("cache:", REAL_TWIN_CACHE_PATH.relative_to(REPO_ROOT))

In [ ]:
def _scale_trace_about_mean(trace: np.ndarray, scale: float) -> np.ndarray:
    tr = np.asarray(trace, dtype=np.float32)
    center = tr.mean(axis=0, keepdims=True)
    return (center + float(scale) * (tr - center)).astype(np.float32)

def _rotate_trace_about_mean(trace: np.ndarray, angle_rad: float = np.pi / 2) -> np.ndarray:
    tr = np.asarray(trace, dtype=np.float32)
    center = tr.mean(axis=0, keepdims=True)
    centered = tr - center
    rot = np.asarray(
        [[np.cos(angle_rad), -np.sin(angle_rad)], [np.sin(angle_rad), np.cos(angle_rad)]],
        dtype=np.float32,
    )
    return (centered @ rot.T + center).astype(np.float32)

def _brownian_trace_matched_to_empirical(trace: np.ndarray, seed: int) -> np.ndarray:
    tr = np.asarray(trace, dtype=np.float32)
    center = tr.mean(axis=0, keepdims=True)
    steps = np.diff(tr, axis=0).astype(np.float64)
    if steps.shape[0] == 0:
        return np.repeat(center, tr.shape[0], axis=0).astype(np.float32)
    rng = np.random.default_rng(seed)
    mu = steps.mean(axis=0)
    cov = np.cov(steps.T) if steps.shape[0] > 1 else np.eye(2) * 1e-10
    cov = np.asarray(cov, dtype=np.float64) + np.eye(2) * 1e-10
    ctrl_steps = rng.multivariate_normal(mu, cov, size=steps.shape[0]).astype(np.float32)
    ctrl = np.vstack([np.zeros((1, 2), dtype=np.float32), np.cumsum(ctrl_steps, axis=0)])
    ctrl -= ctrl.mean(axis=0, keepdims=True)
    target_rms = np.sqrt(np.mean(np.sum((tr - center) ** 2, axis=1)))
    ctrl_rms = np.sqrt(np.mean(np.sum(ctrl ** 2, axis=1)))
    if ctrl_rms > 1e-8:
        ctrl *= float(target_rms / ctrl_rms)
    return (ctrl + center).astype(np.float32)

def real_twin_condition_traces(trace: np.ndarray, seed: int) -> dict[str, np.ndarray]:
    tr = np.asarray(trace, dtype=np.float32)
    center = tr.mean(axis=0, keepdims=True)
    return {
        "static": np.repeat(center, tr.shape[0], axis=0).astype(np.float32),
        "empirical 0.5x": _scale_trace_about_mean(tr, 0.5),
        "empirical 1x": tr.copy(),
        "empirical 2x": _scale_trace_about_mean(tr, 2.0),
        "brownian 1x": _brownian_trace_matched_to_empirical(tr, seed=seed + 101),
        "rotated 1x": _rotate_trace_about_mean(tr, angle_rad=np.pi / 2),
    }

def trace_qc_table(condition_traces: dict[str, np.ndarray]) -> pd.DataFrame:
    rows = []
    for label, trace in condition_traces.items():
        centered = trace - trace.mean(axis=0, keepdims=True)
        steps = np.diff(trace, axis=0)
        step_amp = np.linalg.norm(steps, axis=1) if steps.size else np.zeros(0)
        rows.append({
            "condition": label,
            "rms_displacement_deg": float(np.sqrt(np.mean(np.sum(centered * centered, axis=1)))),
            "path_length_deg": float(step_amp.sum()) if step_amp.size else 0.0,
            "step_rms_deg": float(np.sqrt(np.mean(step_amp * step_amp))) if step_amp.size else 0.0,
            "step_p95_deg": float(np.percentile(step_amp, 95)) if step_amp.size else 0.0,
        })
    return pd.DataFrame(rows)

In [ ]:
def compute_real_twin_demo_cache(
    cache_path: Path,
    *,
    n_units: int,
    t_max: int,
    frame_index: int,
    image_index: int,
    seed: int,
    batch_size: int,
    device_override: str | None,
) -> dict[str, np.ndarray | dict[str, np.ndarray]]:
    from jake.twininfo.common import extract_fixrsvp_eye_traces, load_digital_twin
    from jake.twininfo.lagcube_information import run_lag_cube_rates
    from jake.twininfo.population import build_analysis_population
    from jake.twininfo.retinal_examples import model_lag_cubes_from_image_trace, select_trace_examples
    from jake.twininfo.stimuli import load_natural_images

    model, _model_info, device = load_digital_twin(device=device_override)
    rng = np.random.default_rng(seed)
    population, population_rows = build_analysis_population(
        model,
        N=int(n_units),
        rng=rng,
        selection="top_performance",
        performance_metric="ccnorm",
        grid_position_mode="center",
        deduplicate_units=False,
    )

    eye_traces, durations = extract_fixrsvp_eye_traces(model, min_fix_dur=int(t_max))
    examples = select_trace_examples(
        eye_traces,
        durations,
        t_max=int(t_max),
        n_each=1,
        seed=int(seed),
        stride=8,
    )
    example = next((ex for ex in examples if ex.kind == "fixation"), examples[0])
    (_spec, image) = load_natural_images(1, indices=(int(image_index),))[0]
    condition_traces = real_twin_condition_traces(example.trace, seed=seed)

    labels = list(condition_traces)
    rate_maps = []
    retinal_current_frames = []
    for label in labels:
        trace = condition_traces[label]
        cubes = model_lag_cubes_from_image_trace(image, trace, t_max=int(t_max), crop_center_offset_px=(0.0, 0.0))
        _rates, rate_map = run_lag_cube_rates(
            model,
            population,
            device,
            cubes,
            batch_size=int(batch_size),
            return_rate_map=True,
        )
        rate_maps.append(np.asarray(rate_map[:, :n_units], dtype=np.float32))
        retinal_current_frames.append(np.asarray(cubes[:, 0], dtype=np.float32))

    rate_maps = np.stack(rate_maps, axis=0).astype(np.float32)
    retinal_current_frames = np.stack(retinal_current_frames, axis=0).astype(np.float32)
    frame_index = int(np.clip(frame_index, 0, rate_maps.shape[1] - 1))
    frame_rate_maps = rate_maps[:, frame_index]

    cache_path.parent.mkdir(parents=True, exist_ok=True)
    np.savez_compressed(
        cache_path,
        condition_labels=np.asarray(labels),
        frame_rate_maps=frame_rate_maps,
        rate_maps=rate_maps,
        retinal_current_frames=retinal_current_frames,
        condition_traces=np.stack([condition_traces[label] for label in labels], axis=0),
        frame_index=np.asarray(frame_index, dtype=np.int32),
        t_max=np.asarray(t_max, dtype=np.int32),
        image_index=np.asarray(image_index, dtype=np.int32),
        source_example_id=np.asarray(example.example_id),
        source_trace=np.asarray(example.trace, dtype=np.float32),
        population_session=np.asarray([str(row.get("session_name", "")) for row in population_rows[:n_units]]),
        population_neuron_id=np.asarray([int(row.get("original_neuron_id", -1)) for row in population_rows[:n_units]], dtype=np.int32),
        population_score=np.asarray([float(row.get("performance_score", np.nan)) for row in population_rows[:n_units]], dtype=np.float32),
    )
    return load_real_twin_demo_cache(cache_path)

def load_real_twin_demo_cache(cache_path: Path) -> dict[str, np.ndarray | dict[str, np.ndarray]]:
    d = np.load(cache_path, allow_pickle=True)
    labels = [str(x) for x in d["condition_labels"]]
    frame_rate_maps = np.asarray(d["frame_rate_maps"], dtype=np.float32)
    condition_traces = np.asarray(d["condition_traces"], dtype=np.float32)
    maps_by_condition = {label: frame_rate_maps[i] for i, label in enumerate(labels)}
    traces_by_condition = {label: condition_traces[i] for i, label in enumerate(labels)}
    return {
        "labels": np.asarray(labels),
        "maps_by_condition": maps_by_condition,
        "traces_by_condition": traces_by_condition,
        "rate_maps": np.asarray(d["rate_maps"], dtype=np.float32),
        "retinal_current_frames": np.asarray(d["retinal_current_frames"], dtype=np.float32),
        "frame_index": np.asarray(d["frame_index"]).item(),
        "image_index": np.asarray(d["image_index"]).item(),
        "source_example_id": str(np.asarray(d["source_example_id"]).item()),
        "population_session": np.asarray(d["population_session"]),
        "population_neuron_id": np.asarray(d["population_neuron_id"]),
        "population_score": np.asarray(d["population_score"], dtype=np.float32),
    }

In [ ]:
demo_maps = None
DEMO_SOURCE = ""
real_twin_demo = None

if RUN_REAL_TWIN_DEMO:
    try:
        if REAL_TWIN_CACHE_PATH.exists() and not FORCE_REAL_TWIN_RECOMPUTE:
            real_twin_demo = load_real_twin_demo_cache(REAL_TWIN_CACHE_PATH)
        else:
            real_twin_demo = compute_real_twin_demo_cache(
                REAL_TWIN_CACHE_PATH,
                n_units=N_UNITS_TO_SHOW,
                t_max=REAL_TWIN_T_MAX,
                frame_index=REAL_TWIN_FRAME_INDEX,
                image_index=REAL_TWIN_IMAGE_INDEX,
                seed=RNG_SEED,
                batch_size=REAL_TWIN_BATCH_SIZE,
                device_override=REAL_TWIN_DEVICE,
            )
        demo_maps = real_twin_demo["maps_by_condition"]
        DEMO_SOURCE = "real digital twin"
        print(f"Loaded {DEMO_SOURCE} maps from {REAL_TWIN_CACHE_PATH.relative_to(REPO_ROOT)}")
        print("map shapes:", {label: arr.shape for label, arr in demo_maps.items()})
        print("source example:", real_twin_demo["source_example_id"], "image:", real_twin_demo["image_index"], "frame:", real_twin_demo["frame_index"])
        display(trace_qc_table(real_twin_demo["traces_by_condition"]).style.format(precision=4))
    except Exception as exc:
        print("Real twin demo failed; synthetic fallback will be used.")
        print(type(exc).__name__, exc)
else:
    print("Real twin demo skipped by SSI_WALKTHROUGH_SKIP_REAL_TWIN=1.")

## 4b. Synthetic Fallback

The cells below define a tiny synthetic fallback only for cases where the checkpoint or data are unavailable. In normal use, `demo_maps` has already been filled with real digital-twin activation maps by the previous section.

In [ ]:
def make_demo_activation_maps(n_units: int = 12, height: int = 42, width: int = 42, seed: int = 7) -> dict[str, np.ndarray]:
    rng = np.random.default_rng(seed)
    yy, xx = np.mgrid[-1.0:1.0:complex(height), -1.0:1.0:complex(width)]

    base_maps = []
    unit_params = []
    for _ in range(n_units):
        cx, cy = rng.uniform(-0.45, 0.45, size=2)
        sx, sy = rng.uniform(0.14, 0.34, size=2)
        theta = rng.uniform(0, np.pi)
        carrier = np.cos(2 * np.pi * rng.uniform(1.2, 2.8) * (np.cos(theta) * xx + np.sin(theta) * yy))
        envelope = np.exp(-0.5 * (((xx - cx) / sx) ** 2 + ((yy - cy) / sy) ** 2))
        rate = 0.10 + rng.uniform(0.5, 1.6) * envelope * (1.0 + 0.25 * carrier)
        base_maps.append(np.clip(rate, 1e-5, None))
        unit_params.append((cx, cy, sx, sy, theta))

    base = np.asarray(base_maps, dtype=np.float32)

    def history_field(angle: float, scale: float, wiggle: float = 0.0) -> np.ndarray:
        direction = np.cos(angle) * xx + np.sin(angle) * yy
        orthogonal = -np.sin(angle) * xx + np.cos(angle) * yy
        ridge = np.exp(-0.5 * (orthogonal / 0.22) ** 2) * (1.0 + 0.35 * direction)
        swirl = np.sin(2.5 * np.pi * (direction + wiggle * orthogonal))
        return scale * (0.55 * ridge + 0.20 * swirl)

    condition_specs = {
        "static": (0.0, 0.00),
        "empirical 0.5x": (0.15, 0.45),
        "empirical 1x": (0.15, 0.75),
        "empirical 2x": (0.15, 1.15),
        "brownian 1x": (1.00, 0.78),
        "rotated 1x": (0.15 + np.pi / 2, 0.75),
    }

    maps_by_condition = {}
    for label, (angle, strength) in condition_specs.items():
        maps = []
        for unit, base_map in enumerate(base):
            phase = (unit % 5) * 0.09
            signed = history_field(angle + 0.12 * unit, strength, wiggle=phase)
            gain = 1.0 + signed
            adapted = base_map * np.clip(gain, 0.15, None)
            adapted += 0.018 * unit + 0.02 * strength
            maps.append(np.clip(adapted, 1e-5, None))
        maps_by_condition[label] = np.asarray(maps, dtype=np.float32)
    return maps_by_condition

if demo_maps is None:
    demo_maps = make_demo_activation_maps(N_UNITS_TO_SHOW, seed=RNG_SEED)
    DEMO_SOURCE = "synthetic fallback"
print("demo source:", DEMO_SOURCE)
{label: maps.shape for label, maps in demo_maps.items()}

## 5. The Activation Maps

Each panel is one unit on the same current frame. Across rows, only the history/control condition changes.

In [ ]:
def robust_limits(arrays: list[np.ndarray], q: tuple[float, float] = (1.0, 99.0)) -> tuple[float, float]:
    pooled = np.concatenate([np.ravel(a) for a in arrays])
    lo, hi = np.percentile(pooled, q)
    if not np.isfinite(lo) or not np.isfinite(hi) or hi <= lo:
        lo, hi = float(np.nanmin(pooled)), float(np.nanmax(pooled) + 1e-6)
    return float(lo), float(hi)

def plot_condition_rows(maps_by_condition: dict[str, np.ndarray], units: np.ndarray | None = None):
    labels = list(maps_by_condition)
    units = np.arange(N_UNITS_TO_SHOW) if units is None else np.asarray(units)
    vmin, vmax = robust_limits([maps_by_condition[label][units] for label in labels])
    fig, axes = plt.subplots(len(labels), len(units), figsize=(1.15 * len(units), 1.15 * len(labels)), constrained_layout=True)
    axes = np.asarray(axes).reshape(len(labels), len(units))
    for row, label in enumerate(labels):
        for col, unit in enumerate(units):
            ax = axes[row, col]
            ax.imshow(maps_by_condition[label][unit], cmap="magma", vmin=vmin, vmax=vmax, interpolation="nearest")
            ax.set_xticks([])
            ax.set_yticks([])
            if row == 0:
                ax.set_title(f"u{unit}", fontsize=7)
            if col == 0:
                ax.set_ylabel(label, rotation=0, ha="right", va="center", fontsize=8)
    return fig

plot_condition_rows(demo_maps);

Step through one condition at a time if the full row grid is too dense.

In [ ]:
def plot_unit_maps(maps: np.ndarray, title: str, units: np.ndarray | None = None):
    units = np.arange(min(N_UNITS_TO_SHOW, maps.shape[0])) if units is None else np.asarray(units)
    cols = 4
    rows = int(math.ceil(len(units) / cols))
    vmin, vmax = robust_limits([maps[units]])
    fig, axes = plt.subplots(rows, cols, figsize=(2.0 * cols, 2.0 * rows), constrained_layout=True)
    axes = np.asarray(axes).ravel()
    for ax in axes:
        ax.set_axis_off()
    for ax, unit in zip(axes, units, strict=False):
        ax.imshow(maps[unit], cmap="magma", vmin=vmin, vmax=vmax, interpolation="nearest")
        ax.set_title(f"unit {unit}", fontsize=8)
    fig.suptitle(title, y=1.02)
    return fig

plot_unit_maps(demo_maps["static"], "Static/no-motion history");

In [ ]:
plot_unit_maps(demo_maps["empirical 0.5x"], "Empirical gaze history, scaled 0.5x");

In [ ]:
plot_unit_maps(demo_maps["empirical 1x"], "Empirical gaze history, 1x");

In [ ]:
plot_unit_maps(demo_maps["empirical 2x"], "Empirical gaze history, scaled 2x");

In [ ]:
plot_unit_maps(demo_maps["brownian 1x"], "Brownian motion history, 1x");

In [ ]:
plot_unit_maps(demo_maps["rotated 1x"], "Rotated empirical history, 1x");

## 6. Compute SSI On These Maps

This table is the whole metric in miniature: spatially peaky or structured maps tend to have higher bits/spike than broad, flat maps, after normalizing each unit by its own spatial mean rate.

In [ ]:
demo_ssi = {label: spatial_ssi_single_frame(maps) for label, maps in demo_maps.items()}

rows = []
for label, out in demo_ssi.items():
    rows.append({
        "condition": label,
        "population_bits_per_spike": out["population_bits_per_spike"],
        "mean_unit_bits_per_spike": float(np.mean(out["unit_bits_per_spike"])),
        "median_unit_bits_per_spike": float(np.median(out["unit_bits_per_spike"])),
        "mean_rate": float(np.mean(out["unit_mean_rate"])),
    })
demo_summary = pd.DataFrame(rows)
display(demo_summary.style.format(precision=4))

In [ ]:
unit_demo = pd.DataFrame({label: out["unit_bits_per_spike"] for label, out in demo_ssi.items()})
unit_demo.index.name = "unit"

fig, ax = plt.subplots(figsize=(8.5, 3.2))
x = np.arange(len(unit_demo.index))
for label in unit_demo.columns:
    ax.plot(x, unit_demo[label], marker="o", lw=1.2, ms=3, label=label)
ax.set_xlabel("unit")
ax.set_ylabel("SSI (bits/spike)")
ax.set_title("Single-frame per-unit SSI")
ax.legend(ncol=3, fontsize=7)
fig.tight_layout()

## 7. One Unit, Decomposed

For a chosen unit and condition, we can inspect the raw activation `r(x)`, normalized gain `g(x)`, and local contribution `g(x) log2(g(x))`.

In [ ]:
INSPECT_CONDITION = "empirical 1x"
INSPECT_UNIT = int(np.nanargmax(demo_ssi[INSPECT_CONDITION]["unit_bits_per_spike"]))

r = demo_maps[INSPECT_CONDITION][INSPECT_UNIT]
out = demo_ssi[INSPECT_CONDITION]
gain = out["gain"][INSPECT_UNIT]
local = out["local_bits"][INSPECT_UNIT]
unit_bits = float(out["unit_bits_per_spike"][INSPECT_UNIT])
unit_mean = float(out["unit_mean_rate"][INSPECT_UNIT])

fig, axes = plt.subplots(1, 4, figsize=(10.0, 2.4), constrained_layout=True)
im0 = axes[0].imshow(r, cmap="magma", interpolation="nearest")
axes[0].set_title("r(x)")
plt.colorbar(im0, ax=axes[0], fraction=0.046, pad=0.02)

im1 = axes[1].imshow(gain, cmap="coolwarm", vmin=np.percentile(gain, 2), vmax=np.percentile(gain, 98), interpolation="nearest")
axes[1].set_title("g(x) = r/rbar")
plt.colorbar(im1, ax=axes[1], fraction=0.046, pad=0.02)

im2 = axes[2].imshow(local, cmap="viridis", interpolation="nearest")
axes[2].set_title("g log2(g)")
plt.colorbar(im2, ax=axes[2], fraction=0.046, pad=0.02)

axes[3].hist(gain.ravel(), bins=36, color="0.25")
axes[3].axvline(1.0, color="tab:red", lw=1)
axes[3].set_title("gain histogram")
axes[3].set_xlabel("gain")

for ax in axes[:3]:
    ax.set_xticks([])
    ax.set_yticks([])

fig.suptitle(f"{INSPECT_CONDITION}, unit {INSPECT_UNIT}: SSI={unit_bits:.3f} bits/spike, rbar={unit_mean:.3f}", y=1.08);

## 8. Connect To Cached Twin Results

The cache below contains real digital-twin summaries over BackImage stimuli. It stores compact arrays such as `mean__family__scale` and `ssi_unit_mean__family__scale`. Those are not full `(unit, height, width)` activation maps, but they let us compare the same SSI quantity across the requested gaze-history families without recomputing the twin.

In [ ]:
def first_existing(paths: list[Path]) -> Path | None:
    for path in paths:
        if path.exists():
            return path
    return None

SUMMARY_CACHE_PATH = first_existing(CACHE_CANDIDATES)
if SUMMARY_CACHE_PATH is None:
    display(Markdown("**No cached response_summary_arrays.npz file was found.** The synthetic SSI cells above still run."))
else:
    print(SUMMARY_CACHE_PATH.relative_to(REPO_ROOT))

summary_cache = np.load(SUMMARY_CACHE_PATH) if SUMMARY_CACHE_PATH is not None else None
available_keys = set(summary_cache.files) if summary_cache is not None else set()

def cache_key(metric: str, family: str, scale_id: str) -> str:
    return f"{metric}__{family}__{scale_id}"

cache_availability = []
for label, family, scale_id in REQUESTED_CONDITIONS:
    cache_availability.append({
        "condition": label,
        "mean_key": cache_key("mean", family, scale_id),
        "mean_found": cache_key("mean", family, scale_id) in available_keys,
        "ssi_key": cache_key("ssi_unit_mean", family, scale_id),
        "ssi_found": cache_key("ssi_unit_mean", family, scale_id) in available_keys,
    })
display(pd.DataFrame(cache_availability))

Pick an image row and a set of units. The default selects the image whose requested conditions show the largest across-condition SSI spread, then selects 12 units with large condition-dependent SSI variation.

In [ ]:
def load_condition_arrays(cache, requested_conditions):
    loaded = {}
    if cache is None:
        return loaded
    for label, family, scale_id in requested_conditions:
        mean_name = cache_key("mean", family, scale_id)
        ssi_name = cache_key("ssi_unit_mean", family, scale_id)
        if mean_name in cache.files and ssi_name in cache.files:
            loaded[label] = {
                "mean": np.asarray(cache[mean_name], dtype=np.float32),
                "ssi_unit_mean": np.asarray(cache[ssi_name], dtype=np.float32),
                "mean_key": mean_name,
                "ssi_key": ssi_name,
            }
    return loaded

cached_conditions = load_condition_arrays(summary_cache, REQUESTED_CONDITIONS)
print("loaded conditions:", list(cached_conditions))

if cached_conditions:
    ssi_stack = np.stack([v["ssi_unit_mean"] for v in cached_conditions.values()], axis=0)
    image_spread = np.nanmean(np.nanstd(ssi_stack, axis=0), axis=1)
    IMAGE_ROW = int(np.nanargmax(image_spread))

    unit_spread = np.nanstd(ssi_stack[:, IMAGE_ROW, :], axis=0)
    unit_mean_rate = np.nanmean(np.stack([v["mean"][IMAGE_ROW] for v in cached_conditions.values()], axis=0), axis=0)
    candidate_score = unit_spread * np.sqrt(np.maximum(unit_mean_rate, 0))
    UNIT_IDS = np.argsort(candidate_score)[-N_UNITS_TO_SHOW:][::-1]
    print("image row:", IMAGE_ROW)
    print("unit ids:", UNIT_IDS.tolist())
else:
    IMAGE_ROW = 0
    UNIT_IDS = np.arange(N_UNITS_TO_SHOW)

In [ ]:
if cached_conditions:
    cached_unit_rows = []
    for label, arrays in cached_conditions.items():
        for unit in UNIT_IDS:
            cached_unit_rows.append({
                "condition": label,
                "image_row": IMAGE_ROW,
                "unit": int(unit),
                "mean_response": float(arrays["mean"][IMAGE_ROW, unit]),
                "ssi_bits_per_spike": float(arrays["ssi_unit_mean"][IMAGE_ROW, unit]),
            })
    cached_unit_df = pd.DataFrame(cached_unit_rows)
    display(cached_unit_df.head(18).style.format(precision=4))
else:
    cached_unit_df = pd.DataFrame()
    display(Markdown("No cache-backed unit table was created."))

## 9. Cached Twin SSI Panels

Each panel is one real twin unit from the cache. The x-axis is the requested history/control condition; the line shows cached unit-mean SSI for the selected BackImage row.

In [ ]:
if not cached_unit_df.empty:
    labels = list(cached_conditions)
    label_to_x = {label: i for i, label in enumerate(labels)}
    cols = 4
    rows = int(math.ceil(len(UNIT_IDS) / cols))
    fig, axes = plt.subplots(rows, cols, figsize=(2.45 * cols, 1.9 * rows), sharex=True, sharey=False, constrained_layout=True)
    axes = np.asarray(axes).ravel()
    for ax in axes:
        ax.set_axis_off()
    for ax, unit in zip(axes, UNIT_IDS, strict=False):
        sub = cached_unit_df[cached_unit_df["unit"] == int(unit)].copy()
        sub["x"] = sub["condition"].map(label_to_x)
        sub = sub.sort_values("x")
        ax.set_axis_on()
        ax.plot(sub["x"], sub["ssi_bits_per_spike"], marker="o", lw=1.3, color="tab:blue")
        ax.set_title(f"unit {int(unit)}", fontsize=8)
        ax.set_xticks(range(len(labels)))
        ax.set_xticklabels([s.replace(" ", "\n") for s in labels], rotation=0, fontsize=6)
        ax.set_ylabel("SSI", fontsize=7)
    fig.suptitle(f"Cached twin unit-mean SSI, image row {IMAGE_ROW}", y=1.02)
else:
    display(Markdown("No cached panels to plot."))

In [ ]:
if not cached_unit_df.empty:
    pivot = cached_unit_df.pivot(index="unit", columns="condition", values="ssi_bits_per_spike")
    pivot = pivot[[label for label in cached_conditions if label in pivot.columns]]
    fig, ax = plt.subplots(figsize=(7.8, 3.7))
    im = ax.imshow(pivot.values, aspect="auto", cmap="viridis")
    ax.set_yticks(range(len(pivot.index)))
    ax.set_yticklabels(pivot.index)
    ax.set_xticks(range(len(pivot.columns)))
    ax.set_xticklabels(pivot.columns, rotation=35, ha="right")
    ax.set_xlabel("history/control condition")
    ax.set_ylabel("unit")
    ax.set_title("Cached unit SSI heatmap")
    plt.colorbar(im, ax=ax, label="bits/spike")
    fig.tight_layout()
else:
    display(Markdown("No cached heatmap to plot."))

## 10. Cached Response vs SSI

SSI is not just mean rate. This scatter is useful when explaining why an activation map can become more informative even if the mean response changes only modestly.

In [ ]:
if not cached_unit_df.empty:
    fig, ax = plt.subplots(figsize=(5.6, 4.2))
    for label, sub in cached_unit_df.groupby("condition", sort=False):
        ax.scatter(sub["mean_response"], sub["ssi_bits_per_spike"], s=32, label=label, alpha=0.78)
    ax.set_xlabel("cached mean response")
    ax.set_ylabel("cached unit-mean SSI (bits/spike)")
    ax.set_title("Mean response and spatial information separate")
    ax.legend(fontsize=7, ncol=2)
    fig.tight_layout()
else:
    display(Markdown("No cached scatter to plot."))

## 11. Inspect Or Swap The Rate-Map Cache

The real-twin demo cache stores both the plotted single-frame maps and the full short time series. You can point `FULL_RATE_MAP_CACHE` at another `.npz` with rate-map-like arrays to inspect or reuse it.

In [ ]:
FULL_RATE_MAP_CACHE: Path | None = REAL_TWIN_CACHE_PATH if REAL_TWIN_CACHE_PATH.exists() else None

def load_rate_map_like_arrays(path: Path) -> dict[str, np.ndarray]:
    loaded = np.load(path)
    out = {}
    for key in loaded.files:
        arr = np.asarray(loaded[key])
        if arr.ndim in {4, 5} and np.issubdtype(arr.dtype, np.number):
            out[key] = arr.astype(np.float32)
    return out

if FULL_RATE_MAP_CACHE is None:
    display(Markdown("No rate-map cache selected. Set `FULL_RATE_MAP_CACHE = Path(...)` to inspect another cache."))
elif not FULL_RATE_MAP_CACHE.exists():
    display(Markdown(f"Selected full rate-map cache does not exist: `{FULL_RATE_MAP_CACHE}`"))
else:
    full_maps = load_rate_map_like_arrays(FULL_RATE_MAP_CACHE)
    print({key: arr.shape for key, arr in full_maps.items()})

## 12. Takeaways

- SSI is a spatial-pattern metric: it normalizes out each unit's mean rate before asking how nonuniform the spatial activation is.
- A single plotted frame can still depend on history, because the twin receives a lagged/recurrent retinal movie.
- The demo panels now come from the real digital twin: one natural image frame, 12 high-performing readout units, and six history/control trajectories.
- The existing BackImage SSI caches still provide the broader cached summary comparison across static, empirical, scaled empirical, Brownian, and rotated conditions.

## 13. Logarithmic Eye-Scale Sweep — SSI vs Motion Amplitude

For each scale `s` in a log-spaced range, the empirical trace is contracted or expanded about its mean (`s=0` = static, `s=1` = empirical). The full `T_MAX`-frame history is played into the twin, and SSI is computed on a **single activation-map snapshot** at the last frame. No temporal averaging — the twin's built-in lag history does the integration.

Expected shape: SSI should rise from the static baseline as motion diversifies the retinal input, then potentially turn over when motion is fast enough to smear the activation map toward uniformity.

In [ ]:
SWEEP_SCALES = np.concatenate([[0.0], np.logspace(-1.5, 1.0, 24)])  # [0, ~0.03x … 10x], 25 points
SWEEP_CACHE_PATH = REPO_ROOT / "notebooks/cache/spatial_ssi_log_sweep.npz"

print(f"sweep scales ({len(SWEEP_SCALES)}): {np.round(SWEEP_SCALES, 3).tolist()}")

In [ ]:
def compute_log_sweep_cache(
    cache_path: Path,
    *,
    sweep_scales: np.ndarray,
    source_trace: np.ndarray,
    image_index: int,
    n_units: int,
    t_max: int,
    frame_index: int,
    seed: int,
    batch_size: int,
    device_override: str | None,
) -> dict:
    from jake.twininfo.common import load_digital_twin
    from jake.twininfo.lagcube_information import run_lag_cube_rates
    from jake.twininfo.population import build_analysis_population
    from jake.twininfo.retinal_examples import model_lag_cubes_from_image_trace
    from jake.twininfo.stimuli import load_natural_images

    model, _info, device = load_digital_twin(device=device_override)
    rng = np.random.default_rng(seed)
    population, _ = build_analysis_population(
        model, N=n_units, rng=rng,
        selection="top_performance", performance_metric="ccnorm",
        grid_position_mode="center", deduplicate_units=False,
    )
    (_spec, image) = load_natural_images(1, indices=(image_index,))[0]

    frame_maps = []
    for i, s in enumerate(sweep_scales):
        trace = _scale_trace_about_mean(source_trace, float(s))
        cubes = model_lag_cubes_from_image_trace(
            image, trace, t_max=t_max, crop_center_offset_px=(0.0, 0.0)
        )
        _rates, rate_map = run_lag_cube_rates(
            model, population, device, cubes,
            batch_size=batch_size, return_rate_map=True,
        )
        # Single snapshot at frame_index — no temporal averaging
        frame_maps.append(np.asarray(rate_map[frame_index, :n_units], dtype=np.float32))
        print(f"  scale {s:.3f} ({i+1}/{len(sweep_scales)}) done")

    frame_maps = np.stack(frame_maps, axis=0)  # (n_scales, n_units, H, W)
    cache_path.parent.mkdir(parents=True, exist_ok=True)
    np.savez_compressed(
        cache_path,
        sweep_scales=sweep_scales.astype(np.float32),
        frame_maps=frame_maps,
        frame_index=np.int32(frame_index),
        image_index=np.int32(image_index),
    )
    return {"sweep_scales": sweep_scales, "frame_maps": frame_maps}


sweep_data = None
if RUN_REAL_TWIN_DEMO and real_twin_demo is not None:
    source_trace = np.asarray(
        real_twin_demo["traces_by_condition"]["empirical 1x"], dtype=np.float32
    )
    print("Computing log-scale sweep (this loads the twin checkpoint)…")
    sweep_data = compute_log_sweep_cache(
        SWEEP_CACHE_PATH,
        sweep_scales=SWEEP_SCALES,
        source_trace=source_trace,
        image_index=REAL_TWIN_IMAGE_INDEX,
        n_units=N_UNITS_TO_SHOW,
        t_max=REAL_TWIN_T_MAX,
        frame_index=REAL_TWIN_FRAME_INDEX,
        seed=RNG_SEED,
        batch_size=REAL_TWIN_BATCH_SIZE,
        device_override=REAL_TWIN_DEVICE,
    )
    print("Done.")
else:
    print("Sweep requires the real twin demo; skipped (RUN_REAL_TWIN_DEMO=False or demo failed).")

In [ ]:
if sweep_data is not None:
    scales = sweep_data["sweep_scales"]          # (n_scales,)
    frame_maps = sweep_data["frame_maps"]        # (n_scales, n_units, H, W)

    # SSI at each scale: single-frame snapshot per unit
    unit_ssi = np.stack(
        [spatial_ssi_single_frame(frame_maps[i])["unit_bits_per_spike"] for i in range(len(scales))],
        axis=0,
    )  # (n_scales, n_units)
    pop_ssi = np.stack(
        [spatial_ssi_single_frame(frame_maps[i])["population_bits_per_spike"] for i in range(len(scales))],
        axis=0,
    )  # (n_scales,)

    # Reference lines at the discrete conditions computed earlier
    ref_scales = {"static": 0.0, "0.5x": 0.5, "1x": 1.0, "2x": 2.0}
    ref_ssi = {
        label: spatial_ssi_single_frame(demo_maps[f"empirical {label}"])["population_bits_per_spike"]
        if f"empirical {label}" in demo_maps else None
        for label in ["0.5x", "1x", "2x"]
    }
    ref_ssi["static"] = spatial_ssi_single_frame(demo_maps["static"])["population_bits_per_spike"]

    # x values for log axis: treat static (s=0) as a small sentinel so it sits left of the log range
    x_log = np.where(scales > 0, scales, scales[scales > 0].min() * 0.4)

    fig, axes = plt.subplots(1, 2, figsize=(10.5, 3.8), constrained_layout=True)

    # --- Left: population SSI ---
    ax = axes[0]
    ax.plot(x_log, pop_ssi, color="tab:blue", lw=2, marker="o", ms=4, zorder=3, label="population (spike-weighted)")
    for label, s in ref_scales.items():
        v = ref_ssi.get(label)
        if v is not None:
            ax.axvline(max(s, x_log.min()), color="0.6", lw=0.8, ls="--")
            ax.text(max(s, x_log.min()), ax.get_ylim()[1] if ax.get_ylim()[1] > 0 else 1,
                    label, fontsize=7, ha="center", va="bottom", color="0.5")
    ax.set_xscale("log")
    ax.set_xlabel("Eye-motion scale (relative to empirical)")
    ax.set_ylabel("SSI (bits/spike)")
    ax.set_title("Population SSI vs motion amplitude")
    ax.legend(fontsize=8)

    # --- Right: per-unit SSI (light lines + median ribbon) ---
    ax = axes[1]
    for u in range(unit_ssi.shape[1]):
        ax.plot(x_log, unit_ssi[:, u], color="tab:blue", lw=0.7, alpha=0.25)
    med = np.median(unit_ssi, axis=1)
    p25, p75 = np.percentile(unit_ssi, 25, axis=1), np.percentile(unit_ssi, 75, axis=1)
    ax.fill_between(x_log, p25, p75, alpha=0.18, color="tab:blue")
    ax.plot(x_log, med, color="tab:blue", lw=2, label="median unit SSI")
    ax.set_xscale("log")
    ax.set_xlabel("Eye-motion scale (relative to empirical)")
    ax.set_ylabel("SSI (bits/spike)")
    ax.set_title(f"Per-unit SSI ({unit_ssi.shape[1]} units) — IQR ribbon")
    ax.legend(fontsize=8)

    # mark static sentinel on both axes
    for ax in axes:
        ax.axvline(x_log[0], color="0.7", lw=0.8, ls=":")
        ax.text(x_log[0], ax.get_ylim()[0], "static", fontsize=7, ha="center", va="bottom", color="0.5")

    fig.suptitle("SSI as a function of eye-motion amplitude (log sweep, single-frame snapshot)", y=1.02)
else:
    display(Markdown("Sweep data not available — run the compute cell above with the real twin."))

## 14. Along-Edge vs Across-Edge Motion — Anisotropic Conditions

Uses the axis-conditioned trace logic from Panel 4D (`axis_conditioned_traces.matched_axis_trace_pair`). Two families tested:

- **1D projection** — source trace projected onto the edge axis (along) or its perpendicular (across); motion is exactly 1D, the extreme ellipse with infinite aspect ratio.
- **Anisotropic Brownian ellipse** — 2D Brownian walk with a covariance ellipse aligned to the edge axis. `ELLIPSE_ASPECT_RATIO` = σ_major / σ_minor; set to 1 for isotropic, larger values for stronger anisotropy.

`EDGE_AXIS_DEG` is the local image edge orientation in degrees (0 = horizontal). Can be read from the Panel 4D thumbnail CSV or set by hand.

In [ ]:
from declan.axis_conditioned_backimage_trajectory_observer.axis_conditioned_traces import (
    axis_unit,
    axis_perp,
    matched_axis_trace_pair,
)

# --- Knobs ---
# Edge axis from Panel 4D thumbnail; override by hand if you want a specific angle.
_D_THUMBNAIL_CSV = (
    REPO_ROOT
    / "declan/figure4_active_sensing_atlas/figures/panel_D/story_options"
    / "4D_row17_row18_visible_rail_fit_orientation_values.csv"
)
if _D_THUMBNAIL_CSV.exists():
    _thumb = pd.read_csv(_D_THUMBNAIL_CSV)
    EDGE_AXIS_DEG = float(_thumb["visible_rail_fit_axis_deg"].iloc[0])
    print(f"EDGE_AXIS_DEG loaded from Panel 4D thumbnail CSV: {EDGE_AXIS_DEG:.1f}°")
else:
    EDGE_AXIS_DEG = 45.0  # fallback: 45° diagonal
    print(f"Panel 4D thumbnail CSV not found; using EDGE_AXIS_DEG = {EDGE_AXIS_DEG}°")

ELLIPSE_ASPECT_RATIO = 4.0   # σ_major / σ_minor for anisotropic Brownian

print(f"edge axis: {EDGE_AXIS_DEG:.1f}°  |  ellipse aspect ratio: {ELLIPSE_ASPECT_RATIO:.1f}x")

In [ ]:
def _anisotropic_brownian_trace(
    source_trace: np.ndarray,
    *,
    axis_deg: float,
    aspect_ratio: float,
    relation: str,
    seed: int,
) -> np.ndarray:
    """Anisotropic Brownian walk whose covariance ellipse is aligned to axis_deg.

    `relation` = "parallel"   → major axis along edge  (σ_along > σ_across)
    `relation` = "orthogonal" → major axis across edge (σ_across > σ_along)
    `aspect_ratio`            = σ_major / σ_minor

    RMS displacement is matched to the source trace so amplitude is controlled.
    """
    tr = np.asarray(source_trace, dtype=np.float64)
    center = tr.mean(axis=0, keepdims=True)
    T = tr.shape[0]

    u = axis_unit(float(axis_deg))   # along-edge unit vector
    v = axis_perp(float(axis_deg))   # across-edge unit vector

    r = float(aspect_ratio)
    if relation == "parallel":
        sigma_u, sigma_v = r, 1.0      # elongated along edge
    elif relation == "orthogonal":
        sigma_u, sigma_v = 1.0, r      # elongated across edge
    else:
        raise ValueError(f"relation must be 'parallel' or 'orthogonal', got {relation!r}")

    # Covariance in gaze coordinates: C = sigma_u^2 * uu^T + sigma_v^2 * vv^T
    cov = sigma_u**2 * np.outer(u, u) + sigma_v**2 * np.outer(v, v)
    cov += np.eye(2) * 1e-12

    rng = np.random.default_rng(seed)
    steps = rng.multivariate_normal(np.zeros(2), cov, size=T - 1)
    walk = np.vstack([np.zeros((1, 2)), np.cumsum(steps, axis=0)])
    walk -= walk.mean(axis=0, keepdims=True)

    # Match RMS displacement to source
    target_rms = float(np.sqrt(np.mean(np.sum((tr - center) ** 2, axis=1))))
    walk_rms = float(np.sqrt(np.mean(np.sum(walk ** 2, axis=1))))
    if walk_rms > 1e-10:
        walk *= target_rms / walk_rms

    return (walk + center).astype(np.float32)


def axis_conditioned_conditions(
    source_trace: np.ndarray,
    *,
    edge_axis_deg: float,
    aspect_ratio: float,
    seed: int,
) -> dict[str, np.ndarray]:
    """Build the four axis-conditioned conditions from a source fixation trace."""
    tr = np.asarray(source_trace, dtype=np.float32)

    # 1D projections via matched_axis_trace_pair (Panel 4D logic)
    pair = matched_axis_trace_pair(
        tr,
        edge_axis_deg=float(edge_axis_deg),
        template_mode="same_dominant_projection",
        scale=1.0,
    )
    return {
        "along edge (1D)":    pair["parallel"]["trace"].astype(np.float32),
        "across edge (1D)":   pair["orthogonal"]["trace"].astype(np.float32),
        "along edge (ellipse)":  _anisotropic_brownian_trace(
            tr, axis_deg=edge_axis_deg, aspect_ratio=aspect_ratio,
            relation="parallel", seed=seed + 200,
        ),
        "across edge (ellipse)": _anisotropic_brownian_trace(
            tr, axis_deg=edge_axis_deg, aspect_ratio=aspect_ratio,
            relation="orthogonal", seed=seed + 201,
        ),
    }


# Preview the traces
if real_twin_demo is not None:
    _src = np.asarray(real_twin_demo["traces_by_condition"]["empirical 1x"], dtype=np.float32)
    axis_cond_traces = axis_conditioned_conditions(
        _src, edge_axis_deg=EDGE_AXIS_DEG, aspect_ratio=ELLIPSE_ASPECT_RATIO, seed=RNG_SEED,
    )
    display(trace_qc_table(axis_cond_traces).style.format(precision=4))

    fig, axes = plt.subplots(1, len(axis_cond_traces), figsize=(3.2 * len(axis_cond_traces), 3.0),
                             constrained_layout=True)
    u_along = axis_unit(EDGE_AXIS_DEG)
    u_across = axis_perp(EDGE_AXIS_DEG)
    for ax, (label, tr) in zip(axes, axis_cond_traces.items()):
        ax.scatter(tr[:, 0], tr[:, 1], c=np.arange(len(tr)), cmap="viridis", s=4, alpha=0.7)
        # draw axis arrows from centre
        ctr = tr.mean(axis=0)
        scale_arrow = 0.06
        ax.annotate("", ctr + u_along * scale_arrow, ctr,
                    arrowprops=dict(arrowstyle="->", color="green", lw=1.4))
        ax.annotate("", ctr + u_across * scale_arrow, ctr,
                    arrowprops=dict(arrowstyle="->", color="purple", lw=1.4))
        ax.set_title(label, fontsize=9)
        ax.set_xlabel("X (deg)")
        ax.set_ylabel("Y (deg)")
        ax.set_aspect("equal")
        ax.grid(True, alpha=0.3)
    fig.suptitle(f"Axis-conditioned traces (edge axis = {EDGE_AXIS_DEG:.1f}°, green=along, purple=across)",
                 y=1.02)
else:
    display(Markdown("Axis-conditioned trace preview requires the real twin demo."))

In [ ]:
# Compute rate maps and SSI for the four axis-conditioned conditions.
# Uses the same single-snapshot approach as sections 13 and 4.

axis_cond_maps = None

if RUN_REAL_TWIN_DEMO and real_twin_demo is not None:
    from jake.twininfo.common import load_digital_twin
    from jake.twininfo.lagcube_information import run_lag_cube_rates
    from jake.twininfo.population import build_analysis_population
    from jake.twininfo.retinal_examples import model_lag_cubes_from_image_trace
    from jake.twininfo.stimuli import load_natural_images

    _src = np.asarray(real_twin_demo["traces_by_condition"]["empirical 1x"], dtype=np.float32)
    axis_cond_traces = axis_conditioned_conditions(
        _src, edge_axis_deg=EDGE_AXIS_DEG, aspect_ratio=ELLIPSE_ASPECT_RATIO, seed=RNG_SEED,
    )

    model, _info, device = load_digital_twin(device=REAL_TWIN_DEVICE)
    rng = np.random.default_rng(RNG_SEED)
    population, _ = build_analysis_population(
        model, N=N_UNITS_TO_SHOW, rng=rng,
        selection="top_performance", performance_metric="ccnorm",
        grid_position_mode="center", deduplicate_units=False,
    )
    (_spec, image) = load_natural_images(1, indices=(REAL_TWIN_IMAGE_INDEX,))[0]

    axis_cond_maps = {}
    for label, trace in axis_cond_traces.items():
        cubes = model_lag_cubes_from_image_trace(
            image, trace, t_max=REAL_TWIN_T_MAX, crop_center_offset_px=(0.0, 0.0)
        )
        _rates, rate_map = run_lag_cube_rates(
            model, population, device, cubes,
            batch_size=REAL_TWIN_BATCH_SIZE, return_rate_map=True,
        )
        axis_cond_maps[label] = np.asarray(rate_map[REAL_TWIN_FRAME_INDEX, :N_UNITS_TO_SHOW], dtype=np.float32)
        print(f"  {label}: done")
    print("Axis-conditioned rate maps computed.")
else:
    display(Markdown("Axis-conditioned SSI requires the real twin demo."))

In [ ]:
if axis_cond_maps is not None:
    # SSI for each axis-conditioned condition + key baselines for reference
    ref_labels = ["static", "empirical 1x", "brownian 1x"]
    all_maps = {**{k: demo_maps[k] for k in ref_labels if k in demo_maps}, **axis_cond_maps}
    all_ssi = {label: spatial_ssi_single_frame(maps) for label, maps in all_maps.items()}

    # --- Summary table ---
    rows = []
    for label, out in all_ssi.items():
        rows.append({
            "condition": label,
            "population_bits_per_spike": out["population_bits_per_spike"],
            "mean_unit_bits_per_spike": float(np.mean(out["unit_bits_per_spike"])),
        })
    display(pd.DataFrame(rows).style.format(precision=4))

    # --- Bar chart: population SSI ---
    labels_plot = list(all_ssi)
    pop_vals = [all_ssi[l]["population_bits_per_spike"] for l in labels_plot]
    colors = ["#aaaaaa"] * len(ref_labels) + ["#2f8f6a", "#2f8f6a", "#8064a2", "#8064a2"]

    fig, axes = plt.subplots(1, 2, figsize=(10.5, 3.8), constrained_layout=True)

    ax = axes[0]
    bars = ax.bar(range(len(labels_plot)), pop_vals, color=colors, width=0.6, edgecolor="white")
    ax.set_xticks(range(len(labels_plot)))
    ax.set_xticklabels([l.replace(" ", "\n") for l in labels_plot], fontsize=8)
    ax.set_ylabel("SSI (bits/spike)")
    ax.set_title("Population SSI — axis-conditioned vs baselines")
    ax.axhline(all_ssi["empirical 1x"]["population_bits_per_spike"],
               color="0.4", lw=1, ls="--", label="empirical 1x")
    ax.legend(fontsize=8)

    # --- Per-unit scatter: along vs across (1D) ---
    ax = axes[1]
    if "along edge (1D)" in all_ssi and "across edge (1D)" in all_ssi:
        x = all_ssi["along edge (1D)"]["unit_bits_per_spike"]
        y = all_ssi["across edge (1D)"]["unit_bits_per_spike"]
        ax.scatter(x, y, s=40, color="tab:blue", alpha=0.8, zorder=3)
        lim = [min(x.min(), y.min()) * 0.95, max(x.max(), y.max()) * 1.05]
        ax.plot(lim, lim, "k--", lw=1, alpha=0.5)
        ax.set_xlabel("SSI — along edge (1D)")
        ax.set_ylabel("SSI — across edge (1D)")
        ax.set_title("Per-unit: along vs across (1D projection)")
        ax.set_xlim(lim); ax.set_ylim(lim)

        n_above = int(np.sum(x > y))
        ax.text(0.05, 0.93, f"along > across: {n_above}/{len(x)} units",
                transform=ax.transAxes, fontsize=8)

    if "along edge (ellipse)" in all_ssi and "across edge (ellipse)" in all_ssi:
        x_e = all_ssi["along edge (ellipse)"]["unit_bits_per_spike"]
        y_e = all_ssi["across edge (ellipse)"]["unit_bits_per_spike"]
        ax.scatter(x_e, y_e, s=40, color="tab:orange", alpha=0.8, zorder=3,
                   label=f"ellipse (AR={ELLIPSE_ASPECT_RATIO:.0f}x)")
        ax.legend(fontsize=8)

    fig.suptitle(
        f"Axis-conditioned SSI (edge = {EDGE_AXIS_DEG:.1f}°, ellipse AR = {ELLIPSE_ASPECT_RATIO:.0f}x)",
        y=1.02,
    )
else:
    display(Markdown("Axis-conditioned maps not available."))